# CLM: architecture tests (experiments 1, 2, 3b, 4 and 6)
Notebook settings: **Accelerator = GPU T4 x2** (Qwen3-8B is split across the two GPUs), **Internet = On**.
First run with `QUICK = True`. If everything passes, set it to `False` and use *Save Version → Save & Run All*.
Same texts, options and seeds as the Laya and Lev notebooks. There is no Experiment 2b here:
CLM never places the options in a sequence, so there is no position or distance to measure.

Instead of the vLLM server used in the CLM repository, Qwen3-8B runs through `transformers` with the same recipe:
last token of the last layer, normalized. This keeps the three models in the same runtime environment.

In [ ]:
!pip install -q -U transformers umap-learn
!pip install -q --no-deps "contrastive-lm @ git+https://github.com/Contrastive-LM/CLM@bb42c6c5bf914fd449bed2f6ca65be80602cb1f7"

In [ ]:
# 0. Configuration
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"  # the official MLP checkpoint is a dict saved with torch.save

import json, time, re, sys, platform, logging, zipfile
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

QUICK = True
DTYPE = "float16"   # the T4 has no native bf16; if the sanity check fails (NaN), try "bfloat16"

CFG = dict(
    seed=42,
    exp1_n_options=[2, 10, 50, 150, 500], exp1_n_examples=100, exp1_latency_repeats=30,
    exp2_n_options=[4, 10, 20, 50, 77],   exp2_n_examples=100, exp2_n_perms=5,
    exp4_per_intent=20, exp6_per_intent=25, embed_batch=32,
)
if QUICK:
    CFG.update(exp1_n_examples=5, exp1_latency_repeats=5, exp2_n_examples=5, exp2_n_perms=3,
               exp4_per_intent=2, exp6_per_intent=4)
RUN = dict(exp1=True, exp2=True, exp3b=True, exp4=True, exp6=True)

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT = ROOT / "clm_bench"
for d in ["results", "logs", "figures"]:
    (OUT / d).mkdir(parents=True, exist_ok=True)
RUN_ID = time.strftime("%Y%m%d-%H%M%S") + ("-quick" if QUICK else "")
SFX = "_quick" if QUICK else ""

log = logging.getLogger("clm_bench"); log.setLevel(logging.INFO); log.handlers.clear()
for h in [logging.FileHandler(OUT / "logs" / f"run_{RUN_ID}.log"), logging.StreamHandler(sys.stdout)]:
    h.setFormatter(logging.Formatter("%(asctime)s | %(message)s", "%H:%M:%S")); log.addHandler(h)
log.info(f"run_id={RUN_ID} out={OUT}")

In [ ]:
# 1. Model, metadata and sanity check
import transformers, datasets, clm
from transformers import AutoModel, AutoTokenizer
from clm.embedder import Embedder, l2
from clm.engine import Engine
from clm.heads import download as download_head
from clm.schema import state_text, build_pairs

ENC_ID = "Qwen/Qwen3-8B"
N_GPU = torch.cuda.device_count()
t0 = time.perf_counter()
ENC_TOK = AutoTokenizer.from_pretrained(ENC_ID)
ENC_TOK.padding_side = "right"
ENC = AutoModel.from_pretrained(ENC_ID, dtype=getattr(torch, DTYPE), device_map="auto",
                                max_memory={i: "12GiB" for i in range(N_GPU)}).eval()

def sync():
    for i in range(N_GPU): torch.cuda.synchronize(i)

class LocalEmbedder(Embedder):
    """CLM's Embedder (same per-text cache), with Qwen running here instead of on a vLLM server.
    Recipe from the repository: raw text without special tokens, last token, last layer, normalized."""
    def __init__(self, batch=32, max_tokens=2048):
        super().__init__(url="local://qwen3-8b", max_tokens=max_tokens, batch=batch)
    def _fetch(self, texts):
        ids = [ENC_TOK(t, add_special_tokens=False)["input_ids"][-self.max_tokens:] or
               ENC_TOK(" ", add_special_tokens=False)["input_ids"] for t in texts]
        width = max(len(x) for x in ids)
        pad = ENC_TOK.pad_token_id or 0
        first = next(ENC.parameters()).device
        x = torch.tensor([r + [pad] * (width - len(r)) for r in ids], device=first)
        m = torch.tensor([[1] * len(r) + [0] * (width - len(r)) for r in ids], device=first)
        with torch.no_grad():
            h = ENC(input_ids=x, attention_mask=m).last_hidden_state
        last = torch.tensor([len(r) - 1 for r in ids], device=h.device)
        v = h[torch.arange(len(ids), device=h.device), last].float().cpu().numpy()
        return [l2(row) for row in v], sum(len(r) for r in ids)

emb = LocalEmbedder(batch=CFG["embed_batch"])
HEAD_PATH = download_head()
engine = Engine(embedder=emb, checkpoint=HEAD_PATH, device="cuda:0", action_cache="0")
MODELS = {"clm": "clm-latest", "raw": "clm-raw"}   # raw = cosine in Qwen's raw space, without the MLPs

meta = dict(run_id=RUN_ID, quick=QUICK, cfg=CFG, python=sys.version.split()[0], platform=platform.platform(),
            torch=torch.__version__, transformers=transformers.__version__, datasets=datasets.__version__,
            clm=clm.__version__, clm_commit="bb42c6c5", encoder=ENC_ID, dtype=DTYPE, gpus=N_GPU,
            gpu_names=[torch.cuda.get_device_name(i) for i in range(N_GPU)],
            device_map={str(k): str(v) for k, v in getattr(ENC, "hf_device_map", {}).items()},
            head_cfg=engine.heads["clm-latest"].cfg, head_scale=engine.heads["clm-latest"].scale,
            head_params=engine.heads["clm-latest"].n_params, load_seconds=round(time.perf_counter() - t0, 1))

def timed(state, questions, model="clm-latest"):
    sync(); t = time.perf_counter()
    r = engine.answer(state, questions, model=model)
    sync(); return r, (time.perf_counter() - t) * 1000

sanity_state = "Customer: my invoice was charged twice and nobody answers the phone!"
sanity_q = {"department": {"type": "choice", "instructions": "Which team should handle this?",
                           "criteria": {"billing": "Charges, invoices, refunds", "technical": "Bugs and outages"}}}
emb.cache.clear()
r0, cold_ms = timed(sanity_state, sanity_q)
lat = [timed(sanity_state, sanity_q)[1] for _ in range(20)]
a = r0["answers"]["department"]
meta["sanity"] = dict(choice=a["choice"], probabilities=a["probabilities"], cold_ms=cold_ms,
                      all_cached_ms_median=float(np.median(lat)), readme_reference={"billing": 0.93878})
(OUT / "results" / f"run_meta_{RUN_ID}.json").write_text(json.dumps(meta, indent=2, default=str))
log.info(f"meta: {json.dumps({k: meta[k] for k in ['gpu_names', 'dtype', 'transformers', 'head_scale', 'head_params']})}")
log.info(f"sanity: {meta['sanity']}")
assert not any(np.isnan(v) for v in a["probabilities"].values()), "NaN in the probabilities: change DTYPE"
assert a["choice"] == "billing", "Sanity check failed: the README example did not return 'billing'"

In [ ]:
# 2. Utilities: resumable JSONL logs and timed calls
def jdefault(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return str(o)

class JsonlLog:
    """One line per prediction. Keys already written are skipped (resume within a session)."""
    def __init__(self, name):
        self.path = OUT / "results" / f"{name}{SFX}.jsonl"
        self.done = set()
        if self.path.exists():
            for line in self.path.open():
                try: self.done.add(json.loads(line)["key"])
                except Exception: pass
    def write(self, rec):
        rec = {"run_id": RUN_ID, **rec}
        with self.path.open("a") as f: f.write(json.dumps(rec, default=jdefault) + "\n")
        self.done.add(rec["key"])
    def df(self):
        return pd.read_json(self.path, lines=True) if self.path.exists() else pd.DataFrame()

def humanize(label):
    return re.sub(r"^in:", "", str(label).strip().lower()).replace("_", " ").strip()

def choice_q(criteria, instructions):
    return {"q": {"type": "choice", "instructions": instructions, "criteria": criteria}}

def tv(p, q):
    keys = set(p) | set(q)
    return 0.5 * sum(abs(p.get(k, 0) - q.get(k, 0)) for k in keys)

def savefig(name):
    plt.tight_layout(); plt.savefig(OUT / "figures" / f"{name}{SFX}.png", dpi=160); plt.show()

def ask(state, qs, reg="clm"):
    r, ms = timed(state, qs, MODELS[reg])
    a = r["answers"]["q"]
    return a["choice"], dict(a["probabilities"]), ms, r["usage"]["input_tokens"]

In [ ]:
# 3. Data: same label pool as Laya and Lev (banking77 + clinc150 + MASSIVE = 284)
from datasets import load_dataset
b77 = load_dataset("mteb/banking77", split="test").to_pandas()[["text", "label_text"]]
B77_LABELS = sorted(b77["label_text"].unique())
POOL, seen, POOL_SOURCES = [], set(), {}
def add_to_pool(labels, source):
    n0 = len(POOL)
    for lab in labels:
        h = humanize(lab)
        if h and h not in seen and h != "oos":
            seen.add(h); POOL.append((h, source))
    POOL_SOURCES[source] = len(POOL) - n0
add_to_pool(B77_LABELS, "banking77")
add_to_pool(load_dataset("clinc/clinc_oos", "plus", split="test").features["intent"].names, "clinc150")
ds = load_dataset("mteb/amazon_massive_intent", "en", split="test")
add_to_pool(sorted(set(ds["label_text" if "label_text" in ds.column_names else "label"])), "mteb/amazon_massive_intent")
log.info(f"label pool: {len(POOL)} unique, by source: {POOL_SOURCES} (Laya and Lev used 284)")
json.dump({"pool_size": len(POOL), "by_source": POOL_SOURCES, "labels": POOL},
          open(OUT / "results" / "label_pool.json", "w"), indent=1)

INSTR = "What does the customer want? Pick the intent that matches the message."

def option_set(gold_h, n, rng):
    b = [h for h, s in POOL if s == "banking77" and h != gold_h]
    rest = [h for h, s in POOL if s != "banking77"]
    rng.shuffle(b); rng.shuffle(rest)
    opts = [gold_h] + (b + rest)[: n - 1]
    rng.shuffle(opts)
    return opts

def sample_b77(n, seed):
    return b77.sample(n=min(n, len(b77)), random_state=seed).reset_index(drop=True)

In [ ]:
# 4. Experiment 1: scaling with the number of options
# Latency benchmark in three cache scenarios, always with the same option set:
#   cold  = empty cache: Qwen reads the state and all N options
#   warm  = options cached, a new state on every call (the real-use scenario, and the one behind the README's 28 ms)
#   hot   = everything cached: only the MLP and the dot product remain
# Accuracy does not depend on the cache; it is measured with and without the MLPs (regimes clm and raw).
if RUN["exp1"]:
    L1, LB = JsonlLog("exp1_scaling"), JsonlLog("exp1_latency_bench")
    ex = sample_b77(CFG["exp1_n_examples"], CFG["seed"])
    fresh = sample_b77(CFG["exp1_latency_repeats"] + 3, CFG["seed"] + 99).text.tolist()  # fresh states for the benchmark
    for n_req in CFG["exp1_n_options"]:
        n = min(n_req, len(POOL))
        rng = np.random.default_rng(CFG["seed"] + n)
        qs0 = choice_q({o: None for o in option_set(humanize(ex.loc[0, "label_text"]), n, rng)}, INSTR)
        for scen in ["cold", "warm", "hot"]:
            emb.cache.clear()
            ask(fresh[0], qs0)                                     # warms up Qwen and, except in cold, the options
            for r in range(CFG["exp1_latency_repeats"]):
                k = f"{scen}|{n_req}|{r}"
                if k in LB.done: continue
                s = fresh[0] if scen == "hot" else fresh[1 + r]
                if scen == "cold": emb.cache.clear()
                if scen == "warm": emb.cache.pop(state_text(s, INSTR), None)
                _, _, ms, tokens = ask(s, qs0)
                LB.write(dict(key=k, scenario=scen, n_req=n_req, n=n, rep=r, latency_ms=ms, encoded_tokens=tokens))
        for reg in MODELS:
            for i, row in ex.iterrows():
                k = f"{reg}|{n_req}|{i}"
                if k in L1.done: continue
                gold = humanize(row["label_text"])
                rng = np.random.default_rng(CFG["seed"] * 1000 + i * 7 + n)
                qs = choice_q({o: None for o in option_set(gold, n, rng)}, INSTR)
                pred, p, ms, tokens = ask(row["text"], qs, reg)
                L1.write(dict(key=k, regime=reg, n_req=n_req, n=n, idx=i, gold=gold, pred=pred, correct=pred == gold,
                              p_gold=p.get(gold, 0.0), top5=dict(sorted(p.items(), key=lambda kv: -kv[1])[:5])))
        log.info(f"exp1 N={n_req} ok")

    d1, db = L1.df(), LB.df()
    s1 = d1.groupby(["regime", "n_req"]).agg(n_eff=("n", "first"), accuracy=("correct", "mean"), p_gold=("p_gold", "mean"))
    sb = db.groupby(["scenario", "n_req"]).agg(lat_median=("latency_ms", "median"),
                                                lat_p90=("latency_ms", lambda x: np.percentile(x, 90)),
                                                encoded_tokens=("encoded_tokens", "mean"))
    s1.to_csv(OUT / "results" / f"exp1_accuracy{SFX}.csv"); sb.to_csv(OUT / "results" / f"exp1_latency{SFX}.csv")
    display(s1); display(sb)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for scen, g in sb.reset_index().groupby("scenario"):
        ax[0].plot(g.n_req, g.lat_median, "o-", label=scen); ax[2].plot(g.n_req, g.encoded_tokens, "o-", label=scen)
    for reg, g in s1.reset_index().groupby("regime"):
        ax[1].plot(g.n_req, g.accuracy, "o-", label=reg)
    for a_, t in zip(ax, ["median latency (ms)", "accuracy", "tokens read by Qwen per question"]):
        a_.set_xscale("log"); a_.set_xlabel("number of options"); a_.set_title(t); a_.legend()
    savefig("exp1_scaling")

In [ ]:
# 5. Experiment 2: invariance to option order (clm and raw)
# Prediction by construction: zero changes. Each option becomes a vector on its own and options never see each other.
if RUN["exp2"]:
    L2 = JsonlLog("exp2_order")
    ex = sample_b77(CFG["exp2_n_examples"], CFG["seed"] + 1)
    for reg in MODELS:
        for n in CFG["exp2_n_options"]:
            for i, row in ex.iterrows():
                gold = humanize(row["label_text"])
                rng = np.random.default_rng(CFG["seed"] * 2000 + i * 11 + n)
                base_opts = option_set(gold, n, rng)
                for pi in range(CFG["exp2_n_perms"]):
                    k = f"{reg}|{n}|{i}|{pi}"
                    order = list(base_opts) if pi == 0 else list(rng.permutation(base_opts))
                    if k in L2.done: continue
                    pred, p, _, _ = ask(row["text"], choice_q({o: None for o in order}, INSTR), reg)
                    L2.write(dict(key=k, regime=reg, n=n, idx=i, perm=pi, gold=gold, gold_pos=order.index(gold),
                                  pred=pred, correct=pred == gold, probs=p, p_gold=p.get(gold, 0.0)))
            log.info(f"exp2 {reg} N={n} ok")

    d2 = L2.df()
    rows = []
    for (reg, n, i), g in d2.groupby(["regime", "n", "idx"]):
        P = list(g.probs)
        rows.append(dict(regime=reg, n=n, idx=i, flipped=g.pred.nunique() > 1,
                         mean_tv=np.mean([tv(P[a], P[b]) for a in range(len(P)) for b in range(a + 1, len(P))]),
                         p_gold_std=g.p_gold.std(), accuracy=g.correct.mean()))
    s2 = pd.DataFrame(rows).groupby(["regime", "n"]).agg(flip_rate=("flipped", "mean"), mean_tv=("mean_tv", "mean"),
                                                        p_gold_std=("p_gold_std", "mean"), accuracy=("accuracy", "mean"))
    d2["pos_bin"] = pd.cut(d2.gold_pos / (d2.n - 1).clip(lower=1), [-0.01, 0.25, 0.5, 0.75, 1.0],
                           labels=["first", "second", "third", "last"])
    pos = d2.groupby(["regime", "n", "pos_bin"], observed=True).correct.mean().unstack()
    s2.to_csv(OUT / "results" / f"exp2_summary{SFX}.csv"); pos.to_csv(OUT / "results" / f"exp2_position_bias{SFX}.csv")
    display(s2); display(pos)

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for reg, g in s2.reset_index().groupby("regime"):
        ax[0].plot(g.n, g.flip_rate, "o-", label=reg); ax[1].plot(g.n, g.mean_tv, "o-", label=reg)
    for a_, t in zip(ax, ["fraction of texts whose answer changes", "mean distance between distributions (TV)"]):
        a_.set_xlabel("number of options"); a_.set_title(t); a_.legend()
    savefig("exp2_order")

In [ ]:
# 6. Experiment 3b: minimal-edit pairs (same set as Laya and Lev), clm and raw
# Note: here the options have descriptions, and CLM only reads the description (the name never reaches Qwen).
T3B = [
    ("count", "change", "I was charged once for my {0}.", "I was charged twice for my {0}.",
     ("single charge", "a question about one normal charge"), ("duplicate charge", "the same purchase was billed more than once"),
     ("A", "B"), [("coffee",), ("taxi ride",), ("hotel booking",)]),
    ("count", "change", "I entered the wrong PIN once {0}.", "I entered the wrong PIN three times {0}.",
     ("pin question", "a general question about the PIN"), ("card blocked", "the card is blocked after too many wrong PIN attempts"),
     ("A", "B"), [("at the ATM",), ("at a shop",), ("at the petrol station",)]),
    ("count", "change", "I paid for one {0} and received one.", "I paid for two {1} and received one.",
     ("order complete", "everything that was paid for was delivered"), ("missing item", "fewer items were delivered than were paid for"),
     ("A", "B"), [("ticket", "tickets"), ("shirt", "shirts"), ("book", "books")]),
    ("count", "change", "My {0} was paid into my account once this month.", "My {0} was paid into my account twice this month.",
     ("normal payment", "an expected single payment was received"), ("duplicate payment", "the same payment was received more than once"),
     ("A", "B"), [("salary",), ("pension",), ("allowance",)]),
    ("negation", "change", "I want to cancel my {0}.", "I do not want to cancel my {0}.",
     ("cancel", "the customer wants to end it"), ("keep", "the customer wants to keep it"),
     ("A", "B"), [("subscription",), ("membership",), ("plan",)]),
    ("negation", "change", "The {0} arrived.", "The {0} never arrived.",
     ("delivered", "the item was delivered"), ("delivery problem", "the item has not been delivered"),
     ("A", "B"), [("package",), ("card",), ("order",)]),
    ("negation", "change", "I recognise this {0} on my statement.", "I don't recognise this {0} on my statement.",
     ("confirmed", "the customer confirms it is theirs"), ("fraud report", "the customer did not make it"),
     ("A", "B"), [("payment",), ("transfer",), ("charge",)]),
    ("negation", "change", "The {0} works after the update.", "The {0} no longer works after the update.",
     ("working fine", "it works as expected"), ("bug", "it is broken or not working"),
     ("A", "B"), [("app",), ("website",), ("card reader",)]),
    ("numbers", "change", "My balance is ${0} and I want to withdraw ${1}.", "My balance is ${1} and I want to withdraw ${0}.",
     ("sufficient funds", "the balance covers the withdrawal"), ("insufficient funds", "the withdrawal is larger than the balance"),
     ("A", "B"), [("500", "200"), ("900", "300"), ("1000", "50")]),
    ("numbers", "change", "The invoice says ${0} and I was charged ${0}.", "The invoice says ${0} and I was charged ${1}.",
     ("amount correct", "charged exactly the invoice amount"), ("amount mismatch", "charged a different amount than the invoice"),
     ("A", "B"), [("120", "210"), ("45", "54"), ("310", "130")]),
    ("numbers", "change", "I ordered size {0} and received size {0}.", "I ordered size {0} and received size {1}.",
     ("correct item", "received exactly the size ordered"), ("wrong item", "received a different size than ordered"),
     ("A", "B"), [("42", "24"), ("38", "83"), ("46", "64")]),
    ("numbers", "change", "My flight leaves at {0} and I will get to the airport at {1}.", "My flight leaves at {1} and I will get to the airport at {0}.",
     ("on time", "the customer will make the flight"), ("missed flight", "the customer will miss the flight"),
     ("A", "B"), [("9", "7"), ("11", "8"), ("6", "4")]),
    ("roles", "change", "{1} sent money to me.", "I sent money to {0}.",
     ("incoming transfer", "the customer received money"), ("outgoing transfer", "the customer sent money"),
     ("A", "B"), [("the bank", "The bank"), ("my landlord", "My landlord"), ("my friend", "My friend")]),
    ("roles", "change", "I refunded the {0}.", "The {0} refunded me.",
     ("refund issued", "the writer gave money back"), ("refund received", "the writer got money back"),
     ("A", "B"), [("customer",), ("shop",), ("seller",)]),
    ("roles", "change", "I owe {0} money.", "{1} owes me money.",
     ("writer owes", "the writer has a debt to someone"), ("owed to writer", "someone has a debt to the writer"),
     ("A", "B"), [("my brother", "My brother"), ("the shop", "The shop"), ("my landlord", "My landlord")]),
    ("roles", "change", "{1} paid me.", "I paid {0}.",
     ("money received", "the writer received a payment"), ("money paid", "the writer made a payment"),
     ("A", "B"), [("my employer", "My employer"), ("my tenant", "My tenant"), ("my client", "My client")]),
    ("time", "change", "I want to pay before {0}.", "I want to pay after {0}.",
     ("on-time payment", "paying before the deadline"), ("late payment", "paying after the deadline"),
     ("A", "B"), [("the due date",), ("the deadline",), ("the end of the month",)]),
    ("time", "change", "I will travel {0} next week.", "I travelled {0} last week.",
     ("future trip", "about a trip that has not happened yet"), ("past trip", "about a trip that already happened"),
     ("A", "B"), [("abroad",), ("to Spain",), ("to Japan",)]),
    ("time", "change", "I want to stop a {0} that will go out tomorrow.", "I want to stop a {0} that went out yesterday.",
     ("stop future payment", "cancel a payment before it is sent"), ("reverse past payment", "get back a payment that was already sent"),
     ("A", "B"), [("payment",), ("transfer",), ("direct debit",)]),
    ("time", "change", "My {0} renews next month.", "My {0} renewed last month.",
     ("upcoming renewal", "a renewal that has not happened yet"), ("past renewal", "a renewal that already happened"),
     ("A", "B"), [("subscription",), ("membership",), ("insurance",)]),
    ("quantifier", "change", "All of my {0} failed today.", "None of my {0} failed today.",
     ("failed", "they did not go through"), ("went through", "they went through normally"),
     ("A", "B"), [("transfers",), ("payments",), ("top-ups",)]),
    ("quantifier", "change", "Every {0} I tried was declined.", "No {0} I tried was declined.",
     ("declined", "the attempts were rejected"), ("accepted", "the attempts were accepted"),
     ("A", "B"), [("card payment",), ("withdrawal",), ("transfer",)]),
    ("quantifier", "change", "All of my {0} were refunded.", "None of my {0} were refunded.",
     ("refund complete", "the money was returned"), ("refund missing", "the money was not returned"),
     ("A", "B"), [("orders",), ("purchases",), ("tickets",)]),
    ("quantifier", "change", "I received all of my {0}.", "I received none of my {0}.",
     ("all delivered", "everything arrived"), ("nothing delivered", "nothing arrived"),
     ("A", "B"), [("parcels",), ("cards",), ("documents",)]),
    # controls: the edit does not change the meaning, so the answer should not change
    ("paraphrase", "same", "I was charged twice for my {0}.", "I was charged two times for my {0}.",
     ("single charge", "a question about one normal charge"), ("duplicate charge", "the same purchase was billed more than once"),
     ("B", "B"), [("coffee",), ("taxi ride",), ("hotel booking",)]),
    ("paraphrase", "same", "I do not want to cancel my {0}.", "I don't want to cancel my {0}.",
     ("cancel", "the customer wants to end it"), ("keep", "the customer wants to keep it"),
     ("B", "B"), [("subscription",), ("membership",), ("plan",)]),
    ("paraphrase", "same", "My balance is ${1} and I want to withdraw ${0}.", "My balance is {1} dollars and I want to withdraw {0} dollars.",
     ("sufficient funds", "the balance covers the withdrawal"), ("insufficient funds", "the withdrawal is larger than the balance"),
     ("B", "B"), [("500", "200"), ("900", "300"), ("1000", "50")]),
    ("paraphrase", "same", "The {0} never arrived.", "The {0} has not arrived.",
     ("delivered", "the item was delivered"), ("delivery problem", "the item has not been delivered"),
     ("B", "B"), [("package",), ("card",), ("order",)]),
    ("paraphrase", "same", "I sent money to {0}.", "I transferred money to {0}.",
     ("incoming transfer", "the customer received money"), ("outgoing transfer", "the customer sent money"),
     ("B", "B"), [("the bank",), ("my landlord",), ("my friend",)]),
]
OTHER3B, INSTR3B = ("other", "anything else"), "Which option best describes the customer's message?"
if RUN["exp3b"]:
    L3b = JsonlLog("exp3b_minimal_pairs")
    for reg in MODELS:
        for t_id, (cat, kind, ta_t, tb_t, oA, oB, (ga, gb), fills) in enumerate(T3B):
            opt = {"A": oA, "B": oB}
            for f_id, f in enumerate(fills):
                ta, tb = ta_t.format(*f), tb_t.format(*f)
                for order_name, seq in [("fwd", ["A", "B"]), ("rev", ["B", "A"])]:
                    k = f"{reg}|{t_id}|{f_id}|{order_name}"
                    if k in L3b.done: continue
                    crit = {opt[s][0]: opt[s][1] for s in seq}; crit[OTHER3B[0]] = OTHER3B[1]
                    qs = choice_q(crit, INSTR3B)
                    pa, PA, _, _ = ask(ta, qs, reg); pb, PB, _, _ = ask(tb, qs, reg)
                    gA, gB = opt[ga][0], opt[gb][0]
                    L3b.write(dict(key=k, regime=reg, template=t_id, filler=f_id, category=cat, kind=kind,
                                   order=order_name, text_a=ta, text_b=tb, gold_a=gA, gold_b=gB, pred_a=pa, pred_b=pb,
                                   correct_a=pa == gA, correct_b=pb == gB, flipped=pa != pb, tv=tv(PA, PB),
                                   p_gold_b_in_a=PA[gB], p_gold_b_in_b=PB[gB], probs_a=PA, probs_b=PB))
    d = L3b.df()
    d["both_correct"] = d.correct_a & d.correct_b
    d["shift_to_gold_b"] = d.p_gold_b_in_b - d.p_gold_b_in_a
    d["moved_right_way"] = d.shift_to_gold_b > 0
    agg = dict(runs=("key", "count"), acc_a=("correct_a", "mean"), acc_b=("correct_b", "mean"),
               both_correct=("both_correct", "mean"), flip_rate=("flipped", "mean"), mean_tv=("tv", "mean"),
               shift_to_gold_b=("shift_to_gold_b", "mean"), moved_right_way=("moved_right_way", "mean"))
    s3b = d.groupby(["regime", "kind", "category"]).agg(**agg)
    s3b_t = d.groupby(["regime", "kind", "category", "template"]).agg(**agg)
    s3b_t["example_a"] = d.groupby(["regime", "kind", "category", "template"]).text_a.first()
    s3b.to_csv(OUT / "results" / f"exp3b_summary{SFX}.csv"); s3b_t.to_csv(OUT / "results" / f"exp3b_by_template{SFX}.csv")
    display(s3b)
    s3b.loc["clm"].reset_index().set_index("category")[["acc_a", "acc_b", "both_correct", "flip_rate"]].plot.bar(
        figsize=(11, 4), rot=0, ylim=(0, 1))
    plt.title("CLM, minimal edits: accuracy on each side, on both sides, and flip rate (paraphrase = control)")
    savefig("exp3b_minimal_pairs")

In [ ]:
# 7. Experiment 4: labels without meaning (same 12 intents and conditions), clm and raw
# Predictions by construction: original = code_desc (the name is invisible when a description exists);
# misleading always follows the description; code_only stays at chance. Only name_only is not decided in advance.
INTENTS4 = {
    "activate_my_card": "how to activate a newly received card",
    "age_limit": "minimum age required to open or use an account",
    "atm_support": "which cash machines can be used and how",
    "card_swallowed": "a cash machine kept or retained the card",
    "change_pin": "how to change the card PIN code",
    "exchange_rate": "what currency exchange rate is applied",
    "lost_or_stolen_phone": "the phone with the banking app was lost or stolen",
    "terminate_account": "close or delete the account permanently",
    "top_up_limits": "maximum amount that can be added to the account",
    "transaction_charged_twice": "the same payment appears twice",
    "request_refund": "ask for money back for a purchase",
    "edit_personal_details": "update name, address or other personal information",
}
missing = [k for k in INTENTS4 if k not in B77_LABELS]
if missing: log.info(f"exp4: intents missing from banking77, removed: {missing}")
INTENTS4 = {k: v for k, v in INTENTS4.items() if k in B77_LABELS}
I4 = list(INTENTS4)
rng4 = np.random.default_rng(CFG["seed"] + 4)
CODES = {lab: f"K{c}" for lab, c in zip(I4, rng4.choice(np.arange(10, 100), len(I4), replace=False))}
shift = rng4.integers(1, len(I4))
MISLEAD = {lab: humanize(I4[(j + shift) % len(I4)]) for j, lab in enumerate(I4)}  # no key points to its own intent

def criteria_for(cond, order):
    """Returns {shown_key: description} and {shown_key: true intent of the description}."""
    crit, key2int = {}, {}
    for lab in order:
        key = dict(original=humanize(lab), name_only=humanize(lab), code_desc=CODES[lab],
                   code_only=CODES[lab], misleading=MISLEAD[lab])[cond]
        crit[key] = None if cond in ("name_only", "code_only") else INTENTS4[lab]
        key2int[key] = lab
    return crit, key2int
INSTR4 = "Which option matches the customer's request?"
if RUN["exp4"]:
    L4 = JsonlLog("exp4_labels")
    sub = b77[b77.label_text.isin(I4)].groupby("label_text", group_keys=False).apply(
        lambda g: g.sample(n=min(CFG["exp4_per_intent"], len(g)), random_state=CFG["seed"])).reset_index(drop=True)
    for reg in MODELS:
        for i, row in sub.iterrows():
            order = list(np.random.default_rng(CFG["seed"] * 4000 + i).permutation(I4))
            for cond in ["original", "name_only", "code_desc", "code_only", "misleading"]:
                k = f"{reg}|{i}|{cond}"
                if k in L4.done: continue
                crit, key2int = criteria_for(cond, order)
                pred, p, _, _ = ask(row["text"], choice_q(crit, INSTR4), reg)
                L4.write(dict(key=k, regime=reg, idx=i, condition=cond, gold=row["label_text"], pred_key=pred,
                              pred_intent=key2int[pred], correct=key2int[pred] == row["label_text"],
                              followed_key=(cond == "misleading" and pred == humanize(row["label_text"])),
                              p_max=max(p.values())))
    d4 = L4.df()
    s4 = d4.groupby(["regime", "condition"]).agg(n=("idx", "count"), accuracy=("correct", "mean"),
                                                 followed_key=("followed_key", "mean"), p_max=("p_max", "mean"))
    s4["chance"] = 1 / len(I4)
    s4.to_csv(OUT / "results" / f"exp4_summary{SFX}.csv"); display(s4)
    conds = ["original", "name_only", "code_desc", "code_only", "misleading"]
    g4 = s4.loc["clm"].reindex(conds)
    plt.figure(figsize=(8, 4))
    plt.bar(g4.index, g4.accuracy, label="correct by description")
    plt.bar(["misleading"], g4.loc[["misleading"], "followed_key"], bottom=g4.loc[["misleading"], "accuracy"],
            color="C3", label="followed the misleading key")
    plt.axhline(1 / len(I4), ls="--", c="k", label="chance"); plt.ylim(0, 1); plt.legend()
    plt.title("what CLM uses to decide: key or description?")
    savefig("exp4_labels")

In [ ]:
# 8. Experiment 6: X-ray, raw Qwen space against the space after the MLPs
# Points: the texts (state + instruction) and the 12 options, as CLM reads them (by their descriptions).
# In both spaces: cosine accuracy, separation between the correct and wrong options (AUROC),
# and whether the texts cluster by intent (silhouette). The figure projects texts and options together.
if RUN["exp6"]:
    import umap
    from sklearn.metrics import roc_auc_score, silhouette_score

    sub = b77[b77.label_text.isin(I4)].groupby("label_text", group_keys=False).apply(
        lambda g: g.sample(n=min(CFG["exp6_per_intent"], len(g)), random_state=CFG["seed"] + 6)).reset_index(drop=True)
    crit, key2int = criteria_for("original", I4)
    pairs = build_pairs(sub.text.iloc[0], choice_q(crit, INSTR4))["q"]
    opt_keys, opt_texts = pairs[1], pairs[2]
    opt_int = [key2int[k] for k in opt_keys]
    st_texts = [state_text(t, INSTR4) for t in sub.text]
    S_raw, _ = emb.embed(st_texts); O_raw, _ = emb.embed(opt_texts)
    hp = engine.heads["clm-latest"]
    S_head = hp.project_states(S_raw).cpu().numpy(); O_head = hp.project_actions(O_raw).cpu().numpy()
    y = sub.label_text.values
    gold_idx = np.array([opt_int.index(v) for v in y])

    agree = []
    for i, t in enumerate(sub.text):
        pred = ask(t, choice_q(crit, INSTR4), "clm")[0]
        agree.append(key2int[pred] == opt_int[int((S_head[i] @ O_head.T).argmax())])
    log.info(f"exp6: manual computation agrees with engine.answer on {np.mean(agree):.1%} of texts")

    rows6, Z = [], {}
    for name, S, O in [("raw", S_raw, O_raw), ("head", S_head, O_head)]:
        C = S @ O.T                                   # cosines (vectors are already normalized)
        is_gold = np.zeros_like(C, dtype=bool); is_gold[np.arange(len(y)), gold_idx] = True
        others = np.where(is_gold, -np.inf, C).max(1)
        rows6.append(dict(space=name, dim=S.shape[1], accuracy=(C.argmax(1) == gold_idx).mean(),
                          auroc_gold_vs_other=roc_auc_score(is_gold.ravel(), C.ravel()),
                          mean_cos_gold=C[is_gold].mean(), mean_cos_other=C[~is_gold].mean(),
                          mean_margin=(C[np.arange(len(y)), gold_idx] - others).mean(),
                          silhouette_states_by_intent=silhouette_score(S, y, metric="cosine"),
                          mean_cos_state_to_state=float((S @ S.T)[np.triu_indices(len(S), 1)].mean()),
                          mean_cos_option_to_option=float((O @ O.T)[np.triu_indices(len(O), 1)].mean())))
        Z[name] = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=CFG["seed"]).fit_transform(
            np.vstack([S, O]))
    s6 = pd.DataFrame(rows6).set_index("space")
    s6.to_csv(OUT / "results" / f"exp6_summary{SFX}.csv"); display(s6)
    np.savez_compressed(OUT / "results" / f"exp6_vectors{SFX}.npz", S_raw=S_raw, O_raw=O_raw, S_head=S_head, O_head=O_head)
    np.savez_compressed(OUT / "results" / f"exp6_umap{SFX}.npz", **Z)
    pd.DataFrame({"text": sub.text, "intent": y}).to_csv(OUT / "results" / f"exp6_states{SFX}.csv", index=False)
    pd.DataFrame({"key": opt_keys, "text": opt_texts, "intent": opt_int}).to_csv(
        OUT / "results" / f"exp6_options{SFX}.csv", index=False)

    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    cmap = {lab: plt.cm.tab20(j) for j, lab in enumerate(I4)}
    titles = {"raw": "raw Qwen3-8B space (4096-d)", "head": "after the MLPs (512-d)"}
    for j, name in enumerate(["raw", "head"]):
        z = Z[name]; ns = len(y); a_ = ax[j]
        a_.scatter(z[:ns, 0], z[:ns, 1], s=10, c=[cmap[v] for v in y], alpha=.8)
        a_.scatter(z[ns:, 0], z[ns:, 1], s=260, marker="*", c=[cmap[v] for v in opt_int], edgecolors="k")
        a_.set_title(f"{titles[name]}\npoints = texts, stars = options, color = intent "
                     f"(accuracy {s6.loc[name, 'accuracy']:.2f})")
        a_.set_xticks([]); a_.set_yticks([])
    savefig("exp6_xray")

In [ ]:
# 9. Package the results
zpath = ROOT / f"clm_bench_{RUN_ID}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p, p.relative_to(ROOT))
log.info(f"done: {zpath}")